# Building Custom LLM Applications in Dataiku

## Learning Objectives

By completing this notebook, you will:
1. Design end-to-end LLM applications with multiple components
2. Integrate LLMs with Dataiku datasets and APIs
3. Implement caching strategies for performance and cost optimization
4. Build a complete workflow from data ingestion to output
5. Create production-ready applications with monitoring and logging

## Prerequisites

- Module 3: Notebook 01 completed (Python LLM calls)
- Module 1 and 2 completed (Prompts and RAG)
- Understanding of Dataiku datasets and recipes
- Python programming proficiency

## Estimated Time: 70 minutes

---

## Application Architecture

**From prototype to production**: Building a complete LLM application requires more than just API calls.

Key components:
1. **Data ingestion** - Read from Dataiku datasets
2. **Preprocessing** - Clean, validate, format inputs
3. **LLM orchestration** - Prompt engineering, calls, retries
4. **Postprocessing** - Parse, validate, enrich outputs
5. **Output storage** - Write to datasets or APIs
6. **Monitoring** - Log performance, errors, costs

### Key Insight

**The LLM is just one step** in a larger pipeline. Design for the full workflow.

### Example Application: Commodity Report Analyzer

We'll build a complete application that:
- Reads commodity market reports from a dataset
- Extracts structured information using LLMs
- Generates trading signals and risk assessments
- Stores results with confidence scores
- Provides API endpoints for downstream systems

## Setup

Import libraries and setup infrastructure.

In [ ]:
# Purpose: Setup application infrastructure
# Key Concept: Production applications need comprehensive infrastructure

import dataiku
from dataiku.llm import LLM
import json
import time
import hashlib
from typing import Dict, List, Optional, Tuple
from dataclasses import dataclass, asdict
from datetime import datetime
from enum import Enum
import logging

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger('CommodityAnalyzer')

print("✓ Application infrastructure loaded")

## Application Data Models

Define clear data structures for your application.

In [ ]:
# Purpose: Define application data models
# Key Concept: Strong typing prevents errors and clarifies interfaces

class Sentiment(Enum):
    """Market sentiment enumeration."""
    BULLISH = "bullish"
    BEARISH = "bearish"
    NEUTRAL = "neutral"

class Confidence(Enum):
    """Confidence level enumeration."""
    HIGH = "high"
    MEDIUM = "medium"
    LOW = "low"

@dataclass
class MarketReport:
    """Input: Raw commodity market report."""
    report_id: str
    timestamp: str
    source: str
    text: str
    commodity: Optional[str] = None

@dataclass
class MarketAnalysis:
    """Output: Structured analysis of a market report."""
    report_id: str
    commodity: str
    sentiment: Sentiment
    confidence: Confidence
    key_drivers: List[str]
    price_target: Optional[float]
    risk_level: str
    summary: str
    
    # Metadata
    processing_timestamp: str
    tokens_used: int
    latency_ms: float
    model_version: str

@dataclass
class ProcessingMetrics:
    """Metrics for monitoring application performance."""
    total_processed: int
    successful: int
    failed: int
    total_tokens: int
    total_time_ms: float
    avg_latency_ms: float
    error_rate: float

print("✓ Data models defined")

## Caching Layer

Implement intelligent caching to reduce costs and latency.

In [ ]:
# Purpose: Implement response caching
# Key Concept: Caching dramatically reduces costs for repeated queries

class ResponseCache:
    """
    Cache LLM responses to avoid redundant API calls.
    
    Uses content-based hashing for cache keys.
    """
    
    def __init__(self, ttl_seconds: int = 3600):
        self.cache: Dict[str, Tuple[str, float]] = {}  # key -> (response, timestamp)
        self.ttl_seconds = ttl_seconds
        self.hits = 0
        self.misses = 0
    
    def _generate_key(self, prompt: str, model: str, temperature: float) -> str:
        """Generate cache key from prompt parameters."""
        content = f"{prompt}|{model}|{temperature}"
        return hashlib.sha256(content.encode()).hexdigest()
    
    def get(self, prompt: str, model: str, temperature: float) -> Optional[str]:
        """
        Retrieve cached response if available and not expired.
        
        Returns
        -------
        Optional[str]
            Cached response or None
        """
        key = self._generate_key(prompt, model, temperature)
        
        if key in self.cache:
            response, timestamp = self.cache[key]
            age = time.time() - timestamp
            
            if age < self.ttl_seconds:
                self.hits += 1
                logger.info(f"Cache HIT (age: {age:.0f}s)")
                return response
            else:
                # Expired - remove from cache
                del self.cache[key]
        
        self.misses += 1
        logger.info("Cache MISS")
        return None
    
    def put(self, prompt: str, model: str, temperature: float, response: str):
        """Store response in cache."""
        key = self._generate_key(prompt, model, temperature)
        self.cache[key] = (response, time.time())
        logger.debug(f"Cached response (key: {key[:8]}...)")
    
    def clear(self):
        """Clear all cached entries."""
        self.cache.clear()
        self.hits = 0
        self.misses = 0
        logger.info("Cache cleared")
    
    def get_stats(self) -> Dict:
        """Get cache performance statistics."""
        total = self.hits + self.misses
        hit_rate = self.hits / total if total > 0 else 0
        
        return {
            'entries': len(self.cache),
            'hits': self.hits,
            'misses': self.misses,
            'hit_rate': hit_rate,
            'ttl_seconds': self.ttl_seconds
        }

# Test caching
cache = ResponseCache(ttl_seconds=300)

# Simulate cache operations
test_prompt = "What is oil?"
test_model = "claude-sonnet"
test_temp = 0.7

# First call - cache miss
result1 = cache.get(test_prompt, test_model, test_temp)
assert result1 is None, "First call should be cache miss"

# Store response
cache.put(test_prompt, test_model, test_temp, "Oil is a fossil fuel...")

# Second call - cache hit
result2 = cache.get(test_prompt, test_model, test_temp)
assert result2 is not None, "Second call should be cache hit"

print("\nCache Statistics:")
for key, value in cache.get_stats().items():
    print(f"  {key}: {value}")

print("\n✓ Caching system implemented")

## Core Application Engine

Build the main processing engine that orchestrates all components.

In [ ]:
# Purpose: Main application engine
# Key Concept: Orchestrate all components in a maintainable architecture

class CommodityAnalyzerEngine:
    """
    Main engine for commodity report analysis.
    
    Integrates:
    - LLM calls with retry logic
    - Response caching
    - Error handling
    - Performance monitoring
    - Output validation
    """
    
    def __init__(self, llm: LLM, cache: ResponseCache):
        self.llm = llm
        self.cache = cache
        self.model_version = "v1.0"
        
        # System prompt for consistent behavior
        self.system_prompt = """You are an expert commodity market analyst with deep knowledge of:
- Oil, gas, and energy markets
- Precious and industrial metals
- Agricultural commodities
- Supply/demand dynamics
- Geopolitical impacts

Provide data-driven analysis with specific reasoning."""
    
    def analyze_report(self, report: MarketReport) -> MarketAnalysis:
        """
        Analyze a single market report.
        
        Parameters
        ----------
        report : MarketReport
            Input report to analyze
        
        Returns
        -------
        MarketAnalysis
            Structured analysis with metadata
        
        Raises
        ------
        ValueError
            If analysis fails or produces invalid output
        """
        start_time = time.time()
        
        # Step 1: Build analysis prompt
        prompt = self._build_analysis_prompt(report)
        
        # Step 2: Check cache
        cached_response = self.cache.get(prompt, "claude-sonnet", 0.3)
        
        if cached_response:
            response_text = cached_response
            tokens_used = 0  # Cached, no new tokens
            logger.info(f"Using cached analysis for report {report.report_id}")
        else:
            # Step 3: Call LLM with retry logic
            response = self._call_llm_with_retry(prompt, max_retries=3)
            response_text = response.text
            tokens_used = response.usage.total_tokens
            
            # Step 4: Cache the response
            self.cache.put(prompt, "claude-sonnet", 0.3, response_text)
        
        # Step 5: Parse and validate
        analysis_data = self._parse_analysis(response_text)
        
        # Step 6: Create structured output
        latency_ms = (time.time() - start_time) * 1000
        
        analysis = MarketAnalysis(
            report_id=report.report_id,
            commodity=analysis_data.get('commodity', report.commodity or 'Unknown'),
            sentiment=Sentiment(analysis_data['sentiment']),
            confidence=Confidence(analysis_data['confidence']),
            key_drivers=analysis_data['key_drivers'],
            price_target=analysis_data.get('price_target'),
            risk_level=analysis_data['risk_level'],
            summary=analysis_data['summary'],
            processing_timestamp=datetime.now().isoformat(),
            tokens_used=tokens_used,
            latency_ms=latency_ms,
            model_version=self.model_version
        )
        
        logger.info(f"Successfully analyzed report {report.report_id} ({latency_ms:.0f}ms)")
        return analysis
    
    def _build_analysis_prompt(self, report: MarketReport) -> str:
        """Build the analysis prompt."""
        return f"""{self.system_prompt}

Analyze this commodity market report and provide structured insights.

Report:
{report.text}

Provide your analysis in JSON format:
{{
  "commodity": "specific commodity name",
  "sentiment": "bullish" | "bearish" | "neutral",
  "confidence": "high" | "medium" | "low",
  "key_drivers": ["driver 1", "driver 2", "driver 3"],
  "price_target": null or numeric value,
  "risk_level": "low" | "medium" | "high",
  "summary": "2-3 sentence analysis"
}}

JSON:"""
    
    def _call_llm_with_retry(self, prompt: str, max_retries: int = 3):
        """Call LLM with exponential backoff retry."""
        delay = 1.0
        
        for attempt in range(max_retries):
            try:
                response = self.llm.complete(
                    prompt,
                    max_tokens=400,
                    temperature=0.3
                )
                return response
            except Exception as e:
                logger.warning(f"LLM call attempt {attempt + 1} failed: {e}")
                if attempt < max_retries - 1:
                    time.sleep(delay)
                    delay *= 2
                else:
                    raise
    
    def _parse_analysis(self, response_text: str) -> Dict:
        """Parse and validate LLM response."""
        try:
            data = json.loads(response_text)
            
            # Validate required fields
            required = ['commodity', 'sentiment', 'confidence', 'key_drivers', 'risk_level', 'summary']
            for field in required:
                if field not in data:
                    raise ValueError(f"Missing required field: {field}")
            
            # Validate enum values
            if data['sentiment'] not in ['bullish', 'bearish', 'neutral']:
                raise ValueError(f"Invalid sentiment: {data['sentiment']}")
            
            if data['confidence'] not in ['high', 'medium', 'low']:
                raise ValueError(f"Invalid confidence: {data['confidence']}")
            
            return data
            
        except json.JSONDecodeError as e:
            logger.error(f"Failed to parse JSON response: {e}")
            raise ValueError(f"Invalid JSON response: {response_text[:100]}")

print("✓ Analysis engine defined")

## Exercise 1: Test the Analysis Engine

**Task**: Test the engine with various report types and validate outputs.

Create test reports covering:
- Clear bullish signals
- Clear bearish signals
- Mixed/neutral signals
- Different commodity types

In [ ]:
# YOUR CODE HERE

# Initialize engine
llm = LLM('claude-sonnet')
cache = ResponseCache(ttl_seconds=300)
engine = CommodityAnalyzerEngine(llm, cache)

# Create test reports
test_reports = [
    MarketReport(
        report_id="TEST001",
        timestamp="2024-02-03T10:00:00",
        source="EIA",
        text="U.S. crude oil inventories fell by 9.2 million barrels, the largest weekly draw in over a year. OPEC+ production cuts are tightening global supply while demand from China continues to strengthen.",
        commodity="Crude Oil"
    ),
    # YOUR CODE HERE - Add at least 2 more test reports
    MarketReport(
        report_id="TEST002",
        timestamp="2024-02-03T11:00:00",
        source="USDA",
        text="Corn harvest exceeds expectations with yields up 12% year-over-year. Export demand remains weak as global buyers turn to cheaper alternatives. Storage capacity nearing limits in key growing regions.",
        commodity="Corn"
    ),
    MarketReport(
        report_id="TEST003",
        timestamp="2024-02-03T12:00:00",
        source="LME",
        text="Copper prices held steady amid mixed signals. Chinese manufacturing data showed modest growth, but concerns about global recession persist. LME inventories unchanged for third consecutive week.",
        commodity="Copper"
    ),
]

# Process each report
print("=== Testing Analysis Engine ===")
print()

analyses = []
for report in test_reports:
    try:
        analysis = engine.analyze_report(report)
        analyses.append(analysis)
        
        print(f"Report: {report.report_id} ({report.commodity})")
        print(f"  Sentiment: {analysis.sentiment.value} (Confidence: {analysis.confidence.value})")
        print(f"  Key Drivers: {', '.join(analysis.key_drivers)}")
        print(f"  Risk Level: {analysis.risk_level}")
        print(f"  Summary: {analysis.summary}")
        print(f"  Performance: {analysis.latency_ms:.0f}ms, {analysis.tokens_used} tokens")
        print()
    except Exception as e:
        print(f"ERROR processing {report.report_id}: {e}")
        print()

# Test cache effectiveness
print("\n=== Testing Cache ===")
print("Re-processing first report (should hit cache)...")
analysis_cached = engine.analyze_report(test_reports[0])
assert analysis_cached.tokens_used == 0, "Cached analysis should use 0 tokens"
print(f"✓ Cache hit! Latency: {analysis_cached.latency_ms:.0f}ms")

print("\nCache Statistics:")
for key, value in cache.get_stats().items():
    print(f"  {key}: {value}")

# Auto-graded checks
assert len(analyses) == len(test_reports), "Should successfully analyze all reports"
assert all(isinstance(a.sentiment, Sentiment) for a in analyses), "All sentiments should be valid"
assert all(isinstance(a.confidence, Confidence) for a in analyses), "All confidence levels should be valid"
assert all(len(a.key_drivers) > 0 for a in analyses), "All analyses should have key drivers"

print("\n✓ Exercise 1 passed!")

## Batch Processing Pipeline

Process multiple reports efficiently with progress tracking.

In [ ]:
# Purpose: Batch processing with monitoring
# Key Concept: Production pipelines need progress tracking and error isolation

class BatchProcessor:
    """
    Process batches of reports with monitoring and error handling.
    """
    
    def __init__(self, engine: CommodityAnalyzerEngine):
        self.engine = engine
    
    def process_batch(
        self,
        reports: List[MarketReport],
        progress_callback: Optional[callable] = None
    ) -> Tuple[List[MarketAnalysis], ProcessingMetrics]:
        """
        Process a batch of reports.
        
        Parameters
        ----------
        reports : List[MarketReport]
            Reports to process
        progress_callback : callable, optional
            Function to call with progress updates
        
        Returns
        -------
        Tuple[List[MarketAnalysis], ProcessingMetrics]
            Successful analyses and performance metrics
        """
        analyses = []
        errors = []
        total_tokens = 0
        start_time = time.time()
        
        for i, report in enumerate(reports, 1):
            try:
                analysis = self.engine.analyze_report(report)
                analyses.append(analysis)
                total_tokens += analysis.tokens_used
                
                if progress_callback:
                    progress_callback(i, len(reports), report.report_id, 'success')
                    
            except Exception as e:
                error_msg = f"Failed to process {report.report_id}: {str(e)}"
                logger.error(error_msg)
                errors.append({'report_id': report.report_id, 'error': str(e)})
                
                if progress_callback:
                    progress_callback(i, len(reports), report.report_id, 'error')
        
        # Calculate metrics
        total_time_ms = (time.time() - start_time) * 1000
        
        metrics = ProcessingMetrics(
            total_processed=len(reports),
            successful=len(analyses),
            failed=len(errors),
            total_tokens=total_tokens,
            total_time_ms=total_time_ms,
            avg_latency_ms=total_time_ms / len(reports) if reports else 0,
            error_rate=len(errors) / len(reports) if reports else 0
        )
        
        return analyses, metrics

# Progress callback
def print_progress(current: int, total: int, report_id: str, status: str):
    """Print processing progress."""
    status_icon = "✓" if status == 'success' else "✗"
    print(f"[{current}/{total}] {status_icon} {report_id}")

print("✓ Batch processor defined")

## Exercise 2: Build a Complete Pipeline

**Task**: Create a full data processing pipeline that:
1. Reads from a simulated dataset
2. Processes reports in batches
3. Writes results to output format
4. Generates performance report

In [ ]:
# YOUR CODE HERE

def run_analysis_pipeline(input_reports: List[MarketReport]) -> Dict:
    """
    Complete analysis pipeline.
    
    Parameters
    ----------
    input_reports : List[MarketReport]
        Input reports to process
    
    Returns
    -------
    dict
        Pipeline results including analyses, metrics, and summary
    """
    # YOUR CODE HERE
    # 1. Initialize components
    # 2. Process batch
    # 3. Generate summary
    # 4. Return structured results
    
    print("=== Running Analysis Pipeline ===")
    print(f"Input: {len(input_reports)} reports\n")
    
    # Initialize
    llm = LLM('claude-sonnet')
    cache = ResponseCache(ttl_seconds=300)
    engine = CommodityAnalyzerEngine(llm, cache)
    processor = BatchProcessor(engine)
    
    # Process
    print("Processing reports...")
    analyses, metrics = processor.process_batch(input_reports, progress_callback=print_progress)
    
    # Summary statistics
    sentiment_counts = {}
    for analysis in analyses:
        sentiment = analysis.sentiment.value
        sentiment_counts[sentiment] = sentiment_counts.get(sentiment, 0) + 1
    
    # Generate output
    results = {
        'analyses': [asdict(a) for a in analyses],
        'metrics': asdict(metrics),
        'summary': {
            'success_rate': metrics.successful / metrics.total_processed,
            'sentiment_distribution': sentiment_counts,
            'cache_stats': cache.get_stats()
        },
        'pipeline_timestamp': datetime.now().isoformat()
    }
    
    return results

# Create diverse test dataset
pipeline_test_reports = [
    MarketReport(
        report_id="PIPE001",
        timestamp="2024-02-03T09:00:00",
        source="Bloomberg",
        text="Brent crude surged 4.5% on Middle East tensions and unexpected inventory drawdown.",
        commodity="Crude Oil"
    ),
    MarketReport(
        report_id="PIPE002",
        timestamp="2024-02-03T09:15:00",
        source="Reuters",
        text="Natural gas prices plummeted 7% as warm weather forecasts reduced heating demand.",
        commodity="Natural Gas"
    ),
    MarketReport(
        report_id="PIPE003",
        timestamp="2024-02-03T09:30:00",
        source="LBMA",
        text="Gold reached new all-time high above $2,150 on central bank buying and safe-haven demand.",
        commodity="Gold"
    ),
    MarketReport(
        report_id="PIPE004",
        timestamp="2024-02-03T09:45:00",
        source="CME",
        text="Corn futures traded in narrow range with mixed signals from export data and weather patterns.",
        commodity="Corn"
    ),
    MarketReport(
        report_id="PIPE005",
        timestamp="2024-02-03T10:00:00",
        source="LME",
        text="Copper inventories fell sharply as Chinese demand picks up ahead of manufacturing restart.",
        commodity="Copper"
    ),
]

# Run pipeline
pipeline_results = run_analysis_pipeline(pipeline_test_reports)

# Display results
print("\n" + "="*60)
print("PIPELINE RESULTS")
print("="*60)

print("\nMetrics:")
for key, value in pipeline_results['metrics'].items():
    print(f"  {key}: {value}")

print("\nSummary:")
print(f"  Success Rate: {pipeline_results['summary']['success_rate']:.1%}")
print(f"  Sentiment Distribution: {pipeline_results['summary']['sentiment_distribution']}")
print(f"  Cache Hit Rate: {pipeline_results['summary']['cache_stats']['hit_rate']:.1%}")

# Auto-graded checks
assert pipeline_results['metrics']['successful'] > 0, "Should have successful analyses"
assert pipeline_results['summary']['success_rate'] >= 0.8, "Should achieve >=80% success rate"
assert 'analyses' in pipeline_results, "Results should include analyses"

print("\n✓ Exercise 2 passed!")

## Exercise 3: Create an API Endpoint Handler

**Task**: Build a handler for deploying your application as a Dataiku API endpoint.

Your handler should:
- Accept JSON input
- Validate input format
- Process the request
- Return structured JSON response
- Handle errors gracefully

In [ ]:
# YOUR CODE HERE

class APIEndpointHandler:
    """
    Handler for Dataiku API endpoint deployment.
    
    This would be deployed as a Python function endpoint in Dataiku.
    """
    
    def __init__(self):
        # Initialize once when endpoint starts
        self.llm = LLM('claude-sonnet')
        self.cache = ResponseCache(ttl_seconds=600)
        self.engine = CommodityAnalyzerEngine(self.llm, self.cache)
    
    def handle_request(self, request_data: Dict) -> Dict:
        """
        Handle API request.
        
        Expected request format:
        {
            "report_id": "string",
            "text": "report text",
            "commodity": "optional commodity name"
        }
        
        Returns
        -------
        dict
            API response with analysis or error
        """
        try:
            # Validate input
            # YOUR CODE HERE
            required_fields = ['report_id', 'text']
            for field in required_fields:
                if field not in request_data:
                    return {
                        'status': 'error',
                        'error': f"Missing required field: {field}"
                    }
            
            # Create report object
            report = MarketReport(
                report_id=request_data['report_id'],
                timestamp=datetime.now().isoformat(),
                source=request_data.get('source', 'API'),
                text=request_data['text'],
                commodity=request_data.get('commodity')
            )
            
            # Process
            analysis = self.engine.analyze_report(report)
            
            # Return success response
            return {
                'status': 'success',
                'data': {
                    'report_id': analysis.report_id,
                    'commodity': analysis.commodity,
                    'sentiment': analysis.sentiment.value,
                    'confidence': analysis.confidence.value,
                    'key_drivers': analysis.key_drivers,
                    'risk_level': analysis.risk_level,
                    'summary': analysis.summary
                },
                'metadata': {
                    'processing_timestamp': analysis.processing_timestamp,
                    'latency_ms': analysis.latency_ms,
                    'model_version': analysis.model_version
                }
            }
            
        except Exception as e:
            logger.error(f"API request failed: {e}")
            return {
                'status': 'error',
                'error': str(e)
            }

# Test the API handler
api_handler = APIEndpointHandler()

test_requests = [
    # Valid request
    {
        'report_id': 'API_TEST_001',
        'text': 'WTI crude futures jumped 5.2% on supply concerns.',
        'commodity': 'Crude Oil'
    },
    # Missing required field
    {
        'report_id': 'API_TEST_002'
        # Missing 'text' field
    },
    # Valid request without optional fields
    {
        'report_id': 'API_TEST_003',
        'text': 'Gold prices steady ahead of Fed decision.'
    }
]

print("=== Testing API Endpoint ===")
print()

for i, request in enumerate(test_requests, 1):
    print(f"Test {i}:")
    print(f"Request: {json.dumps(request, indent=2)}")
    
    response = api_handler.handle_request(request)
    
    print(f"Response Status: {response['status']}")
    if response['status'] == 'success':
        print(f"  Commodity: {response['data']['commodity']}")
        print(f"  Sentiment: {response['data']['sentiment']}")
        print(f"  Latency: {response['metadata']['latency_ms']:.0f}ms")
    else:
        print(f"  Error: {response['error']}")
    print()

# Auto-graded checks
valid_response = api_handler.handle_request(test_requests[0])
assert valid_response['status'] == 'success', "Valid request should succeed"
assert 'data' in valid_response, "Success response should include data"

invalid_response = api_handler.handle_request(test_requests[1])
assert invalid_response['status'] == 'error', "Invalid request should return error"

print("✓ Exercise 3 passed!")

## Summary

Congratulations! You've built a complete production-ready LLM application. Key takeaways:

1. **Architecture matters** - Design for the full pipeline, not just LLM calls
2. **Strong typing** - Use dataclasses and enums for clarity and safety
3. **Caching** - Dramatically reduces costs for repeated queries
4. **Error handling** - Isolate failures, track success rates
5. **Monitoring** - Log performance, tokens, latency
6. **API design** - Clean interfaces for integration

## Production Deployment Checklist

Before deploying to production:

**Functionality**:
- [ ] All components tested individually
- [ ] End-to-end pipeline validated
- [ ] Edge cases handled gracefully
- [ ] Error messages are actionable

**Performance**:
- [ ] Caching implemented and tested
- [ ] Latency within SLA requirements
- [ ] Token usage optimized
- [ ] Batch processing efficient

**Reliability**:
- [ ] Retry logic with backoff
- [ ] Timeout handling
- [ ] Input validation
- [ ] Output validation

**Observability**:
- [ ] Comprehensive logging
- [ ] Performance metrics tracked
- [ ] Error rates monitored
- [ ] Cost tracking enabled

**Documentation**:
- [ ] API contracts documented
- [ ] Data models specified
- [ ] Error codes defined
- [ ] Example requests provided

## Deployment Patterns in Dataiku

**Python Recipe**: 
- Read from dataset → Process → Write to dataset
- Scheduled execution

**API Endpoint**:
- Real-time request/response
- Deploy APIEndpointHandler as function endpoint

**Scenario**:
- Trigger-based execution
- Automated retraining/updates

**Webapp**:
- Interactive user interface
- Real-time analysis visualization

## Next Steps

- **Module 4**: Deploy and monitor your application in production
- Add RAG capabilities for knowledge-grounded analysis
- Implement user feedback collection
- Build monitoring dashboards